# Quantum Teleportation

This notebook demonstrates the quantum teleportation protocol using Amazon Braket.

Quantum teleportation transfers an unknown quantum state from one qubit (Alice) to another (Bob) using a shared entangled Bell pair and two bits of classical communication. The original state is destroyed in the process, consistent with the no-cloning theorem.

Since Amazon Braket does not support mid-circuit measurement with classical feedforward, this implementation uses the **deferred measurement** principle: classically-controlled gates are replaced by quantum-controlled gates (CNOT and CZ), which produce equivalent results.

**Note:** This implementation creates the Bell pair $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ directly using H and CNOT gates. The algorithm library also provides a `bell_singlet` subroutine (in `bells_inequality`) that prepares the singlet state $|\psi^-\rangle$, which is a different Bell state used for inequality tests.

## References

[1] Bennett, C. H. et al. "Teleporting an unknown quantum state via dual classical and Einstein-Podolsky-Rosen channels." Physical Review Letters 70, 1895 (1993). https://doi.org/10.1103/PhysRevLett.70.1895

[2] Nielsen, M. A. and Chuang, I. L. "Quantum Computation and Quantum Information." Cambridge University Press (2010).

## Run on a local simulator

In [ ]:
import numpy as np
from braket.circuits import Circuit
from braket.devices import LocalSimulator
from braket.tracking import Tracker

from braket.experimental.algorithms.quantum_teleportation import (
    get_quantum_teleportation_results,
    quantum_teleportation_circuit,
    run_quantum_teleportation,
)

tracker = Tracker().start()
device = LocalSimulator()

### Print the circuit

The teleportation circuit for teleporting $|1\rangle$ (prepared with an X gate):

In [ ]:
circ = quantum_teleportation_circuit(Circuit().x(0))
print(circ)

### Teleport different states

We teleport $|0\rangle$, $|1\rangle$, and $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, then verify Bob's qubit matches the original state.

In [ ]:
test_cases = [
    ("|0>", None),
    ("|1>", Circuit().x(0)),
    ("|+>", Circuit().h(0)),
    ("Ry(pi/3)|0>", Circuit().ry(0, np.pi / 3)),
]

names, probs_0, probs_1 = [], [], []
for name, prep in test_cases:
    circ = quantum_teleportation_circuit(prep)
    task = run_quantum_teleportation(circ, device, shots=1000)
    result = get_quantum_teleportation_results(task)
    names.append(name)
    probs_0.append(result["0"])
    probs_1.append(result["1"])
    print(f"Teleport {name}: P(|0>) = {result['0']:.4f}, P(|1>) = {result['1']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, probs_0, width, label=r"$P(|0\rangle)$", color="steelblue")
ax.bar(x + width / 2, probs_1, width, label=r"$P(|1\rangle)$", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("Probability")
ax.set_title("Quantum Teleportation Results (Bob's qubit)")
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Run on a QPU

To run on a QPU, replace `LocalSimulator()` with an `AwsDevice`. The cost for 4 circuits of 1000 shots on IQM Garnet is approximately $7.00 USD.

In [ ]:
# # Uncomment to run on a QPU
# from braket.aws import AwsDevice
# qpu = AwsDevice("arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet")
# circ = quantum_teleportation_circuit(Circuit().x(0))
# task = run_quantum_teleportation(circ, qpu, shots=1000)
# result = get_quantum_teleportation_results(task)
# print(f"QPU result: {result}")

In [ ]:
print("Task Summary")
print(f"{tracker.quantum_tasks_statistics()} \n")
print(
    f"Estimated cost to run this example: "
    f"{tracker.qpu_tasks_cost() + tracker.simulator_tasks_cost():.2f} USD"
)

Note: Charges shown are estimates based on your Amazon Braket simulator and quantum processing unit (QPU) task usage. Estimated charges shown may differ from your actual charges. Estimated charges do not factor in any discounts or credits, and you may experience additional charges based on your use of other services such as Amazon Elastic Compute Cloud (Amazon EC2).